
# cfDNA Fragment Length Distribution

本笔记提供一个可复用的 Python 脚本（使用 `pysam` 读取 BAM）来统计 cfDNA 双端测序数据的片段长度分布，只保留 read1/read2 均成功比对到同一参考位置的成对读段（proper pairs）。

> **流程**
> 1. 安装依赖 `pysam`, `pandas`, `matplotlib`, `tqdm`（见下方单元格）。
> 2. 配置 `bam_path` 等参数。
> 3. 运行计算函数以获得长度分布表格和/或图像。


In [ ]:
%pip install -q pysam pandas matplotlib tqdm

In [ ]:

from __future__ import annotations
from collections import Counter
from pathlib import Path
from typing import Optional, Tuple

import pandas as pd
import pysam
from tqdm import tqdm
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

def compute_fragment_length_distribution(
    bam_path: str | Path,
    min_length: int = 0,
    max_length: Optional[int] = None,
    require_same_start: bool = False,
    verbose: bool = True,
) -> pd.DataFrame:
    """Return a DataFrame with fragment length counts from a paired-end BAM.

    Parameters
    ----------
    bam_path : str or Path
        Input BAM file path (coordinate-sorted, indexed is recommended).
    min_length, max_length : int
        Optional filters applied to the absolute template length (|TLEN|).
    require_same_start : bool
        If True, only retain pairs whose R1/R2 share identical reference id & start.
        By default we only require the pair to be mapped properly.
    verbose : bool
        Whether to show a progress bar while scanning reads.
    """
    bam_path = Path(bam_path)
    if not bam_path.exists():
        raise FileNotFoundError(f"BAM file not found: {bam_path}")

    counter = Counter()

    with pysam.AlignmentFile(bam_path, "rb") as bam:
        iterator = bam.fetch(until_eof=True)
        if verbose:
            iterator = tqdm(iterator, desc="Scanning reads")

        for read in iterator:
            if not read.is_paired or not read.is_read1:
                continue
            if read.is_unmapped or read.mate_is_unmapped:
                continue
            if read.is_secondary or read.is_supplementary or read.is_duplicate:
                continue
            if not read.is_proper_pair:
                continue
            if require_same_start:
                same_chr = read.reference_id == read.next_reference_id
                same_pos = read.reference_start == read.next_reference_start
                if not (same_chr and same_pos):
                    continue

            frag_len = abs(read.template_length)
            if frag_len == 0:
                continue
            if frag_len < min_length:
                continue
            if max_length is not None and frag_len > max_length:
                continue

            counter[frag_len] += 1

    data = (
        pd.Series(counter, name="count")
        .rename_axis("fragment_length")
        .sort_index()
        .reset_index()
    )
    data["fraction"] = data["count"] / data["count"].sum()
    return data

def plot_fragment_distribution(
    distribution: pd.DataFrame,
    *,
    bin_width: int = 5,
    ax: Optional[plt.Axes] = None,
    color: str = "#1f77b4",
) -> Tuple[plt.Figure, plt.Axes]:
    """Plot a smoothed histogram-style chart from the distribution table."""
    if {"fragment_length", "fraction"} - set(distribution.columns):
        raise ValueError("distribution must include 'fragment_length' and 'fraction' columns")

    binned = (
        distribution.assign(bin=lambda df: (df["fragment_length"] // bin_width) * bin_width)
        .groupby("bin", as_index=False)["fraction"].sum()
    )

    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 4))
    else:
        fig = ax.figure

    ax.bar(binned["bin"], binned["fraction"], width=bin_width * 0.9, color=color)
    ax.set_xlabel("Fragment length (bp)")
    ax.set_ylabel("Fraction of properly paired reads")
    ax.set_title("cfDNA fragment length distribution")
    ax.set_xlim(left=0)
    return fig, ax


In [ ]:

# === Example usage ===
# 将路径替换成真实的 BAM 文件及输出文件名
bam_path = "path/to/cfdna.bam"
out_table = Path("fragment_length_distribution.csv")

# 计算长度分布（示例设置）
# distribution = compute_fragment_length_distribution(
#     bam_path=bam_path,
#     min_length=50,
#     max_length=500,
#     require_same_start=False,
# )
# distribution.to_csv(out_table, index=False)
# plot_fragment_distribution(distribution, bin_width=5)
# plt.show()

print("请将示例代码中的路径替换为实际 BAM 文件后再运行。")
